# 결과 평가

In [15]:
# ── 라이브러리 ──
from pathlib import Path
import polars as pl

import numpy as np
from sklearn.metrics import f1_score

from collections import Counter
from sklearn.metrics import confusion_matrix

In [3]:
# ── 경로 ──
RESULTS_DIR = Path("../data/processed/results")
EVAL_PATH = "../data/processed/eval.csv"

# ── eval 정답 불러오기 (점검 기준) ──
eval_df = pl.read_csv(EVAL_PATH)
eval_df = eval_df.rename(lambda col_name: col_name.lstrip("\ufeff"))

In [4]:
# ── LLM 결과 JSONL 8개 읽기 ──
# 앞부분이 모두 null인 컬럼(error 등)도 타입을 맞게 잡도록 전체 행으로 스키마 추론
llm_frames = [
    pl.read_ndjson(result_path, infer_schema_length=None)
    for result_path in sorted(RESULTS_DIR.glob("*.jsonl"))
]
llm_result_df = pl.concat(llm_frames, how="diagonal_relaxed")

# ── kNN 기준선 결과 붙이기 (LLM이 아니라 파싱·에러 개념이 없음) ──
knn_result_df = pl.read_parquet(RESULTS_DIR / "knn_k7.parquet").with_columns(
    pl.lit(True).alias("parse_ok"),
)
all_result_df = pl.concat([llm_result_df, knn_result_df], how="diagonal_relaxed")

# ── 평가용 예측값: 파싱 실패·에러는 "파싱 실패"로 두어 오답 처리 ──
all_result_df = all_result_df.with_columns(
    pl.col("pred_label").fill_null("파싱 실패").alias("pred_for_eval"),
    (pl.col("model") + " | " + pl.col("prompt_type")).alias("combo"),
)
print(all_result_df.shape)

(2700, 24)


## 점검 1: 조합별 건수, review_id 중복, 에러·파싱 실패 수 ──

In [ ]:

check_df = (
    all_result_df.group_by("combo")
    .agg(
        pl.len().alias("rows"),
        pl.col("review_id").n_unique().alias("unique_reviews"),
        pl.col("error").is_not_null().sum().alias("errors"),
        (~pl.col("parse_ok")).sum().alias("parse_fails"),
    )
    .sort("combo")
)
print(check_df)

# ── 점검 2: 모든 조합이 eval 300건과 같은 리뷰를 담고 있는지 ──
eval_review_ids = set(eval_df["review_id"].to_list())
for combo_name, combo_df in all_result_df.group_by("combo"):
    combo_review_ids = set(combo_df["review_id"].to_list())
    if combo_review_ids != eval_review_ids:
        print(f"{combo_name[0]}: 누락 {len(eval_review_ids - combo_review_ids)}건, "
              f"eval에 없는 리뷰 {len(combo_review_ids - eval_review_ids)}건")

# ── 점검 3: 결과 파일의 정답 라벨이 eval.csv와 같은지 (파일 기록 이후 라벨 수정 여부) ──
label_mismatch_df = all_result_df.join(
    eval_df.select("review_id", pl.col("label").alias("eval_label")), on="review_id"
).filter(pl.col("gold_label") != pl.col("eval_label"))
print(f"정답 라벨 불일치: {label_mismatch_df.height}건")

# ── 점검 4: 프롬프트·가이드 버전이 한 가지인지 (LLM 결과만) ──
print(llm_result_df.select("prompt_version", "label_guide_version").unique())

shape: (9, 5)
┌───────────────────────────────┬──────┬────────────────┬────────┬─────────────┐
│ combo                         ┆ rows ┆ unique_reviews ┆ errors ┆ parse_fails │
│ ---                           ┆ ---  ┆ ---            ┆ ---    ┆ ---         │
│ str                           ┆ u32  ┆ u32            ┆ u32    ┆ u32         │
╞═══════════════════════════════╪══════╪════════════════╪════════╪═════════════╡
│ gemma4:e2b-it-qat | fixed     ┆ 300  ┆ 300            ┆ 0      ┆ 0           │
│ gemma4:e2b-it-qat | rag       ┆ 300  ┆ 300            ┆ 0      ┆ 0           │
│ gemma4:e2b-it-qat | random    ┆ 300  ┆ 300            ┆ 0      ┆ 0           │
│ gemma4:e2b-it-qat | zero_shot ┆ 300  ┆ 300            ┆ 0      ┆ 0           │
│ knn | knn_k7                  ┆ 300  ┆ 300            ┆ 0      ┆ 0           │
│ qwen3.5:4b | fixed            ┆ 300  ┆ 300            ┆ 0      ┆ 0           │
│ qwen3.5:4b | rag              ┆ 300  ┆ 300            ┆ 0      ┆ 0           │
│ qwen3.5:4b |

In [6]:
# ── 파싱 실패·에러 원문 확인 ──
failed_df = all_result_df.filter(~pl.col("parse_ok") | pl.col("error").is_not_null())
with pl.Config(fmt_str_lengths=120, tbl_rows=-1):
    print(failed_df.select("combo", "review_id", "error", "raw_output"))

shape: (0, 4)
┌───────┬───────────┬───────┬────────────┐
│ combo ┆ review_id ┆ error ┆ raw_output │
│ ---   ┆ ---       ┆ ---   ┆ ---        │
│ str   ┆ str       ┆ null  ┆ str        │
╞═══════╪═══════════╪═══════╪════════════╡
└───────┴───────────┴───────┴────────────┘


## 점검 2: 주요 결과표 macro-F1(95% 신뢰구간), 정확도, 파싱 실패율, p50·p95 지연시간, 평균 입력 토큰

In [9]:
# ── 설정 ──
LABELS_KO = ["배송 지연", "누락", "제품 오배송", "제품 불량", "위조·비정품", "제품 불만족", "기타"]
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 2026

# ── 조합별로 eval 순서에 맞춘 정답·예측 배열 (부트스트랩 번호가 같은 리뷰를 가리키도록) ──
eval_order_df = eval_df.select("review_id").with_row_index("eval_position")
combo_arrays = {}
for combo_key, combo_df in all_result_df.group_by("combo"):
    aligned_df = eval_order_df.join(combo_df, on="review_id", how="left").sort("eval_position")
    combo_arrays[combo_key[0]] = (
        aligned_df["gold_label"].to_numpy(),
        aligned_df["pred_for_eval"].to_numpy(),
    )


# ── macro-F1: 7개 라벨 기준 ("파싱 실패" 예측은 해당 정답 라벨의 놓친 건으로 계산됨) ──
def macro_f1(gold_labels, pred_labels):
    return f1_score(gold_labels, pred_labels, labels=LABELS_KO, average="macro", zero_division=0)

In [10]:
# ── 부트스트랩 표본 번호: 모든 조합이 같은 번호를 공유 (4단계 짝지은 비교에 재사용) ──
bootstrap_rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_indices = bootstrap_rng.integers(0, eval_df.height, size=(N_BOOTSTRAP, eval_df.height))

# ── 조합별 부트스트랩 macro-F1 1,000개 (9개 조합, 수십 초 걸림) ──
bootstrap_scores = {
    combo_name: np.array([macro_f1(gold_array[sample], pred_array[sample]) for sample in bootstrap_indices])
    for combo_name, (gold_array, pred_array) in combo_arrays.items()
}

In [11]:
# ── 성능 지표: macro-F1, 95% 신뢰구간, 정확도 ──
metric_df = pl.DataFrame([
    {
        "combo": combo_name,
        "macro_f1": macro_f1(gold_array, pred_array),
        "ci_low": np.percentile(bootstrap_scores[combo_name], 2.5),
        "ci_high": np.percentile(bootstrap_scores[combo_name], 97.5),
        "accuracy": (gold_array == pred_array).mean(),
    }
    for combo_name, (gold_array, pred_array) in combo_arrays.items()
])

# ── 속도·토큰 지표 (LLM만) ──
speed_df = (
    all_result_df.filter(pl.col("model") != "knn")
    .group_by("combo")
    .agg(
        pl.col("elapsed_sec").median().alias("p50_sec"),
        pl.col("elapsed_sec").quantile(0.95).alias("p95_sec"),
        pl.col("input_tokens").mean().alias("avg_input_tokens"),
        pl.col("prompt_sec").mean().alias("avg_prompt_sec"),
        pl.col("gen_sec").mean().alias("avg_gen_sec"),
    )
)

# ── 주요 결과표: macro-F1 높은 순 ──
summary_df = metric_df.join(speed_df, on="combo", how="left").sort("macro_f1", descending=True)
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=200):
    print(summary_df)

shape: (9, 10)
┌───────────────────────────────┬──────────┬────────┬─────────┬──────────┬─────────┬─────────┬──────────────────┬────────────────┬─────────────┐
│ combo                         ┆ macro_f1 ┆ ci_low ┆ ci_high ┆ accuracy ┆ p50_sec ┆ p95_sec ┆ avg_input_tokens ┆ avg_prompt_sec ┆ avg_gen_sec │
│ ---                           ┆ ---      ┆ ---    ┆ ---     ┆ ---      ┆ ---     ┆ ---     ┆ ---              ┆ ---            ┆ ---         │
│ str                           ┆ f64      ┆ f64    ┆ f64     ┆ f64      ┆ f64     ┆ f64     ┆ f64              ┆ f64            ┆ f64         │
╞═══════════════════════════════╪══════════╪════════╪═════════╪══════════╪═════════╪═════════╪══════════════════╪════════════════╪═════════════╡
│ qwen3.5:4b | random           ┆ 0.839    ┆ 0.787  ┆ 0.884   ┆ 0.847    ┆ 3.104   ┆ 6.353   ┆ 1060.133         ┆ 2.712          ┆ 0.527       │
│ qwen3.5:4b | rag              ┆ 0.830    ┆ 0.771  ┆ 0.874   ┆ 0.853    ┆ 3.453   ┆ 6.398   ┆ 1062.173         ┆ 2

## 점검 3: 클래스별 F1 (어떤 라벨에서 약한지)

In [12]:
# ── 조합 이름 줄이기: "qwen3.5:4b | random" → "qwen_random" (표 컬럼명용) ──
SHORT_MODEL_NAMES = {"qwen3.5:4b": "qwen", "gemma4:e2b-it-qat": "gemma", "knn": "knn"}


def short_combo_name(combo_name):
    model_name, prompt_type = combo_name.split(" | ")
    return "knn" if model_name == "knn" else f"{SHORT_MODEL_NAMES[model_name]}_{prompt_type}"


# ── 클래스별 F1 표: 행 = 라벨, 열 = 조합 (macro-F1 높은 순) ──
combo_order = summary_df["combo"].to_list()
gold_reference = combo_arrays[combo_order[0]][0]   # 정답은 모든 조합이 같음
class_f1_df = pl.DataFrame({
    "label": LABELS_KO,
    "support": [int((gold_reference == label_ko).sum()) for label_ko in LABELS_KO],   # eval 건수
    **{
        short_combo_name(combo_name): f1_score(
            *combo_arrays[combo_name], labels=LABELS_KO, average=None, zero_division=0
        )
        for combo_name in combo_order
    },
})
with pl.Config(tbl_cols=-1, float_precision=2, tbl_width_chars=250):
    print(class_f1_df)

shape: (7, 11)
┌─────────────┬─────────┬─────────────┬──────────┬──────────────┬────────────┬────────────────┬───────────┬─────────────┬─────────────────┬──────┐
│ label       ┆ support ┆ qwen_random ┆ qwen_rag ┆ gemma_random ┆ qwen_fixed ┆ qwen_zero_shot ┆ gemma_rag ┆ gemma_fixed ┆ gemma_zero_shot ┆ knn  │
│ ---         ┆ ---     ┆ ---         ┆ ---      ┆ ---          ┆ ---        ┆ ---            ┆ ---       ┆ ---         ┆ ---             ┆ ---  │
│ str         ┆ i64     ┆ f64         ┆ f64      ┆ f64          ┆ f64        ┆ f64            ┆ f64       ┆ f64         ┆ f64             ┆ f64  │
╞═════════════╪═════════╪═════════════╪══════════╪══════════════╪════════════╪════════════════╪═══════════╪═════════════╪═════════════════╪══════╡
│ 배송 지연   ┆ 92      ┆ 0.86        ┆ 0.93     ┆ 0.90         ┆ 0.87       ┆ 0.82           ┆ 0.90      ┆ 0.90        ┆ 0.92            ┆ 0.77 │
│ 누락        ┆ 67      ┆ 0.91        ┆ 0.93     ┆ 0.95         ┆ 0.92       ┆ 0.93           ┆ 0.93      ┆ 0

## 점검 4: 조합 간 차이 검정 - 같은 모델에서 RAG vs 고정, 고정 vs 랜덤 등

In [13]:
# ── 비교 목록: (묶음, 조합 A, 조합 B) ──
MODEL_NAMES = ["qwen3.5:4b", "gemma4:e2b-it-qat"]
PROMPT_PAIRS = [("random", "rag"), ("rag", "fixed"), ("fixed", "zero_shot"), ("random", "zero_shot")]

comparisons = []
for model_name in MODEL_NAMES:                       # 프롬프트 효과: 같은 모델 안에서
    for prompt_a, prompt_b in PROMPT_PAIRS:
        comparisons.append(("프롬프트", f"{model_name} | {prompt_a}", f"{model_name} | {prompt_b}"))
for prompt_type in ["zero_shot", "fixed", "random", "rag"]:   # 모델 효과: 같은 프롬프트에서
    comparisons.append(("모델", f"qwen3.5:4b | {prompt_type}", f"gemma4:e2b-it-qat | {prompt_type}"))
best_llm_combo = summary_df.filter(pl.col("combo") != "knn | knn_k7")["combo"][0]
comparisons.append(("기준선", best_llm_combo, "knn | knn_k7"))   # 최고 LLM 조합 vs kNN

In [14]:
# ── 짝지은 부트스트랩: 같은 표본에서 계산한 두 조합의 macro-F1 차이 (A − B) ──
macro_f1_by_combo = dict(zip(metric_df["combo"], metric_df["macro_f1"]))
comparison_rows = []
for group_name, combo_a, combo_b in comparisons:
    score_differences = bootstrap_scores[combo_a] - bootstrap_scores[combo_b]
    ci_low, ci_high = np.percentile(score_differences, [2.5, 97.5])
    if ci_low > 0:
        verdict = "A 우세"
    elif ci_high < 0:
        verdict = "B 우세"
    else:
        verdict = "차이 불확실"   # 신뢰구간이 0을 포함
    comparison_rows.append({
        "group": group_name,
        "combo_a": short_combo_name(combo_a),
        "combo_b": short_combo_name(combo_b),
        "diff": macro_f1_by_combo[combo_a] - macro_f1_by_combo[combo_b],
        "ci_low": ci_low,
        "ci_high": ci_high,
        "a_better_pct": (score_differences > 0).mean() * 100,   # 표본 1,000개 중 A가 높았던 비율
        "verdict": verdict,
    })

comparison_df = pl.DataFrame(comparison_rows)
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=200):
    print(comparison_df)

shape: (13, 8)
┌──────────┬────────────────┬─────────────────┬───────┬────────┬─────────┬──────────────┬─────────────┐
│ group    ┆ combo_a        ┆ combo_b         ┆ diff  ┆ ci_low ┆ ci_high ┆ a_better_pct ┆ verdict     │
│ ---      ┆ ---            ┆ ---             ┆ ---   ┆ ---    ┆ ---     ┆ ---          ┆ ---         │
│ str      ┆ str            ┆ str             ┆ f64   ┆ f64    ┆ f64     ┆ f64          ┆ str         │
╞══════════╪════════════════╪═════════════════╪═══════╪════════╪═════════╪══════════════╪═════════════╡
│ 프롬프트 ┆ qwen_random    ┆ qwen_rag        ┆ 0.010 ┆ -0.033 ┆ 0.054   ┆ 68.600       ┆ 차이 불확실 │
│ 프롬프트 ┆ qwen_rag       ┆ qwen_fixed      ┆ 0.015 ┆ -0.039 ┆ 0.083   ┆ 69.200       ┆ 차이 불확실 │
│ 프롬프트 ┆ qwen_fixed     ┆ qwen_zero_shot  ┆ 0.009 ┆ -0.040 ┆ 0.051   ┆ 66.100       ┆ 차이 불확실 │
│ 프롬프트 ┆ qwen_random    ┆ qwen_zero_shot  ┆ 0.034 ┆ -0.009 ┆ 0.082   ┆ 94.400       ┆ 차이 불확실 │
│ 프롬프트 ┆ gemma_random   ┆ gemma_rag       ┆ 0.038 ┆ -0.007 ┆ 0.101   ┆ 95.200       ┆

## 점검 5: 가장 좋은 조합의 혼동행렬 + 오답 사례

In [20]:
# ── 설정 ──
TARGET_COMBO = "qwen3.5:4b | random"   # 혼동행렬·오답 사례를 볼 조합 (macro-F1 최고)
ERROR_CASES_PATH = "../data/processed/error_cases_qwen_random.csv"

# ── 혼동행렬: 행 = 정답, 열 = 예측 ──
target_gold, target_pred = combo_arrays[TARGET_COMBO]
matrix_counts = confusion_matrix(target_gold, target_pred, labels=LABELS_KO)
confusion_df = pl.DataFrame(
    {"정답 \\ 예측": LABELS_KO}
    | {pred_label: matrix_counts[:, column_index] for column_index, pred_label in enumerate(LABELS_KO)}
)
with pl.Config(tbl_cols=-1, tbl_width_chars=200):
    print(confusion_df)

# ── 자주 헷갈린 (정답 → 예측) 쌍 상위 8개 (대각선 = 정답은 제외) ──
confusion_pairs = [
    (LABELS_KO[gold_index], LABELS_KO[pred_index], int(matrix_counts[gold_index, pred_index]))
    for gold_index in range(len(LABELS_KO))
    for pred_index in range(len(LABELS_KO))
    if gold_index != pred_index and matrix_counts[gold_index, pred_index] > 0
]
for gold_label, pred_label, pair_count in sorted(confusion_pairs, key=lambda pair: -pair[2])[:8]:
    print(f"  {gold_label} → {pred_label}: {pair_count}건")

shape: (7, 8)
┌─────────────┬───────────┬──────┬─────────────┬───────────┬─────────────┬─────────────┬──────┐
│ 정답 \ 예측 ┆ 배송 지연 ┆ 누락 ┆ 제품 오배송 ┆ 제품 불량 ┆ 위조·비정품 ┆ 제품 불만족 ┆ 기타 │
│ ---         ┆ ---       ┆ ---  ┆ ---         ┆ ---       ┆ ---         ┆ ---         ┆ ---  │
│ str         ┆ i64       ┆ i64  ┆ i64         ┆ i64       ┆ i64         ┆ i64         ┆ i64  │
╞═════════════╪═══════════╪══════╪═════════════╪═══════════╪═════════════╪═════════════╪══════╡
│ 배송 지연   ┆ 77        ┆ 3    ┆ 0           ┆ 0         ┆ 0           ┆ 1           ┆ 11   │
│ 누락        ┆ 4         ┆ 59   ┆ 2           ┆ 1         ┆ 0           ┆ 0           ┆ 1    │
│ 제품 오배송 ┆ 1         ┆ 0    ┆ 41          ┆ 1         ┆ 0           ┆ 1           ┆ 1    │
│ 제품 불량   ┆ 1         ┆ 0    ┆ 2           ┆ 27        ┆ 0           ┆ 2           ┆ 0    │
│ 위조·비정품 ┆ 0         ┆ 0    ┆ 0           ┆ 0         ┆ 7           ┆ 0           ┆ 1    │
│ 제품 불만족 ┆ 0         ┆ 0    ┆ 3           ┆ 1         ┆ 0           ┆ 24     

In [21]:
# ── 오답 사례: 리뷰 원문 포함 CSV로 저장 ──
error_cases_df = (
    all_result_df.filter((pl.col("combo") == TARGET_COMBO) & (pl.col("gold_label") != pl.col("pred_for_eval")))
    .join(eval_df.select("review_id", "review_text"), on="review_id")
    .select("review_id", "gold_label", pl.col("pred_for_eval").alias("pred_label"), "review_text")
    .sort("gold_label", "pred_label")
)

with open(ERROR_CASES_PATH, "w", encoding="utf-8-sig") as error_file:   # 엑셀에서도 한글이 안 깨지게
    error_file.write(error_cases_df.write_csv())
print(f"오답 {error_cases_df.height}건 저장 → {ERROR_CASES_PATH}")

오답 46건 저장 → ../data/processed/error_cases_qwen_random.csv


In [22]:
# ── pool 라벨 조회용 ──
pool_df = pl.read_csv("../data/processed/pool.csv")
pool_df = pool_df.rename(lambda col_name: col_name.lstrip("\ufeff"))
pool_label_by_id = dict(zip(pool_df["review_id"], pool_df["label"]))


# ── 예시 7개 라벨의 다수결 (예시는 덜 비슷한 → 가장 비슷한 순이라, 뒤집어서 동률이면 더 비슷한 쪽) ──
def majority_example_label(example_ids):
    example_labels = [pool_label_by_id[example_id] for example_id in reversed(example_ids)]
    return Counter(example_labels).most_common(1)[0][0]


# ── 모델별로 RAG 예측과 예시 라벨 비교, 랜덤 결과와 짝지어 경우 나누기 ──
rag_copy_frames = []
for model_name in MODEL_NAMES:
    model_df = all_result_df.filter(pl.col("model") == model_name)
    random_pred_df = model_df.filter(pl.col("prompt_type") == "random").select(
        "review_id", pl.col("pred_for_eval").alias("pred_random")
    )
    rag_df = (
        model_df.filter(pl.col("prompt_type") == "rag")
        .select("review_id", "gold_label", pl.col("pred_for_eval").alias("pred_rag"), "example_review_ids")
        .join(random_pred_df, on="review_id")
        .with_columns(
            # 예시 목록의 마지막 = 가장 비슷한 예시
            pl.col("example_review_ids").list.last().replace_strict(pool_label_by_id).alias("nearest_label"),
            pl.col("example_review_ids")
            .map_elements(majority_example_label, return_dtype=pl.String)
            .alias("majority_label"),
        )
        .with_columns(
            pl.when((pl.col("pred_rag") == pl.col("gold_label")) & (pl.col("pred_random") == pl.col("gold_label")))
            .then(pl.lit("1. 둘 다 정답"))
            .when(pl.col("pred_random") == pl.col("gold_label"))
            .then(pl.lit("2. RAG만 오답"))
            .when(pl.col("pred_rag") == pl.col("gold_label"))
            .then(pl.lit("3. 랜덤만 오답"))
            .otherwise(pl.lit("4. 둘 다 오답"))
            .alias("case")
        )
    )
    # 경우별 비율 3개 (전체 행도 함께)
    ratio_exprs = [
        pl.len().alias("reviews"),
        (pl.col("pred_rag") == pl.col("nearest_label")).mean().alias("rag_follows_nearest"),
        (pl.col("pred_rag") == pl.col("majority_label")).mean().alias("rag_follows_majority"),
        (pl.col("gold_label") == pl.col("nearest_label")).mean().alias("nearest_is_correct"),
    ]
    case_summary_df = rag_df.group_by("case").agg(ratio_exprs)
    overall_df = rag_df.select(pl.lit("0. 전체").alias("case"), *ratio_exprs)
    rag_copy_frames.append(
        pl.concat([overall_df, case_summary_df], how="vertical_relaxed")
        .with_columns(pl.lit(SHORT_MODEL_NAMES[model_name]).alias("model"))
    )

# ── 결과표 ──
rag_copy_df = pl.concat(rag_copy_frames).select("model", "case", pl.exclude("model", "case")).sort("model", "case")
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=2, tbl_width_chars=200):
    print(rag_copy_df)

shape: (10, 6)
┌───────┬────────────────┬─────────┬─────────────────────┬──────────────────────┬────────────────────┐
│ model ┆ case           ┆ reviews ┆ rag_follows_nearest ┆ rag_follows_majority ┆ nearest_is_correct │
│ ---   ┆ ---            ┆ ---     ┆ ---                 ┆ ---                  ┆ ---                │
│ str   ┆ str            ┆ u32     ┆ f64                 ┆ f64                  ┆ f64                │
╞═══════╪════════════════╪═════════╪═════════════════════╪══════════════════════╪════════════════════╡
│ gemma ┆ 0. 전체        ┆ 300     ┆ 0.62                ┆ 0.66                 ┆ 0.69               │
│ gemma ┆ 1. 둘 다 정답  ┆ 238     ┆ 0.71                ┆ 0.76                 ┆ 0.71               │
│ gemma ┆ 2. RAG만 오답  ┆ 19      ┆ 0.11                ┆ 0.05                 ┆ 0.68               │
│ gemma ┆ 3. 랜덤만 오답 ┆ 11      ┆ 0.45                ┆ 0.36                 ┆ 0.45               │
│ gemma ┆ 4. 둘 다 오답  ┆ 32      ┆ 0.31                ┆ 0.38             

## 번외

In [23]:
# ── 조합별 출력 길이: 출력 생성이 느렸던 이유 ──
# output_tokens는 많은데 raw_output 글자 수가 짧으면 → 화면에 안 보이는 추론(thinking) 토큰일 가능성
output_length_df = (
    all_result_df.filter(pl.col("model") != "knn")
    .group_by("combo")
    .agg(
        pl.col("output_tokens").mean().alias("avg_output_tokens"),
        pl.col("output_tokens").max().alias("max_output_tokens"),
        pl.col("raw_output").str.len_chars().mean().alias("avg_output_chars"),
        pl.col("gen_sec").mean().alias("avg_gen_sec"),
    )
    .sort("combo")
)
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=1, tbl_width_chars=200):
    print(output_length_df)

# ── gemma zero_shot에서 출력 토큰이 가장 많았던 응답 3개 ──
with pl.Config(fmt_str_lengths=400, tbl_rows=-1):
    print(
        all_result_df.filter(pl.col("combo") == "gemma4:e2b-it-qat | zero_shot")
        .sort("output_tokens", descending=True)
        .head(3)
        .select("output_tokens", "raw_output")
    )

shape: (8, 5)
┌───────────────────────────────┬───────────────────┬───────────────────┬──────────────────┬─────────────┐
│ combo                         ┆ avg_output_tokens ┆ max_output_tokens ┆ avg_output_chars ┆ avg_gen_sec │
│ ---                           ┆ ---               ┆ ---               ┆ ---              ┆ ---         │
│ str                           ┆ f64               ┆ i64               ┆ f64              ┆ f64         │
╞═══════════════════════════════╪═══════════════════╪═══════════════════╪══════════════════╪═════════════╡
│ gemma4:e2b-it-qat | fixed     ┆ 64.2              ┆ 954               ┆ 24.3             ┆ 1.7         │
│ gemma4:e2b-it-qat | rag       ┆ 124.2             ┆ 934               ┆ 24.3             ┆ 3.3         │
│ gemma4:e2b-it-qat | random    ┆ 87.6              ┆ 596               ┆ 24.2             ┆ 2.4         │
│ gemma4:e2b-it-qat | zero_shot ┆ 533.9             ┆ 1005              ┆ 33.7             ┆ 14.5        │
│ qwen3.5:4b | fixed   